### Imports

In [1]:
import os
import cv2
import torch
import numpy as np
from utils import *
import pandas as pd
import seaborn as sns
from modules import *
from PIL import Image
import matplotlib.pyplot as plt
from pycocotools.coco import COCO
import utils.for_eye_tracker as et
from scipy.signal import savgol_filter
from transformers import CLIPProcessor, CLIPModel

Current city is copenhagen!
If you wish to change the city, please edit the value in the __init__.py file


/home/s243636/miniforge3/envs/tese/lib/python3.9/site-packages/heartpy/datautils.py:6: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import resource_filename


_________

In [2]:
def process_gaze_to_fixations(csv_path):

    # --- LOAD ---
    df = pd.read_csv(csv_path)

    df = df.rename(columns={
        'Seconds': 'time',
        'GazeX': 'x',
        'GazeY': 'y'
    })

    df['time'] = pd.to_datetime(df['time'])

    df = df[['time', 'x', 'y']]

    # --- CLEAN ---
    df = df.drop_duplicates(subset='time')
    df = df[(df['x'] >= 0) & (df['y'] >= 0)]

    # --- PREPROCESS ---
    df = df.sort_values('time')

    df['x'] = df['x'].interpolate(limit=5)
    df['y'] = df['y'].interpolate(limit=5)

    df = df.dropna(subset=['x', 'y'])

    if len(df) > 5:
        df['x_smooth'] = savgol_filter(df['x'], 5, 2)
        df['y_smooth'] = savgol_filter(df['y'], 5, 2)
    else:
        df['x_smooth'] = df['x']
        df['y_smooth'] = df['y']

    # --- VELOCITY ---
    t_sec = pd.to_datetime(df['time']).astype('int64') / 1e9

    dt = np.diff(t_sec)
    dx = np.diff(df['x_smooth'])
    dy = np.diff(df['y_smooth'])

    dt[dt == 0] = np.nan

    velocity = np.sqrt(dx**2 + dy**2) / dt

    df['velocity'] = np.nan
    df.iloc[1:, df.columns.get_loc('velocity')] = velocity

    # --- IVT CLASSIFICATION ---
    df['event'] = 'fixation'
    df.loc[df['velocity'] > 1000, 'event'] = 'saccade'

    # --- EXTRACT FIXATIONS ---
    fixations = []
    current_fix = None

    for idx in range(len(df)):
        row = df.iloc[idx]
        if row['event'] == 'fixation':
            if current_fix is None:
                current_fix = {
                    'start_time': row['time'],
                    'x': [],
                    'y': []
                }

            current_fix['x'].append(row['x'])
            current_fix['y'].append(row['y'])

        else:
            if current_fix is not None:
                end_time = df.iloc[idx-1]['time']

                fixations.append({
                    'start_time': current_fix['start_time'],
                    'end_time': end_time,
                    'duration': (end_time - current_fix['start_time']).total_seconds(),
                    'x_mean': np.mean(current_fix['x']),
                    'y_mean': np.mean(current_fix['y'])
                })

                current_fix = None

    fixations = pd.DataFrame(fixations)

    if len(fixations) == 0:
        return None

    # --- POST FILTER ---
    fixations['duration_ms'] = fixations['duration'] * 1000
    fixations = fixations[fixations['duration_ms'] >= 100]
    fixations = fixations[fixations['duration_ms'] < 4000]

    fixations["mid_time"] = fixations["start_time"] + (
        fixations["end_time"] - fixations["start_time"]
    ) / 2

    return fixations

Run for all participants

In [3]:
data_root = "/mnt/raid/emotional_data_raquel/supp_mine/gaze"
output_root = "/mnt/raid/emotional_data_raquel/fulldata_analysis_mine/fixations"

for participant in os.listdir(data_root):

    participant_path = os.path.join(data_root, participant)

    for session in os.listdir(participant_path):

        session_path = os.path.join(participant_path, session)
        gaze_csv = os.path.join(session_path, "gaze.csv")

        if not os.path.exists(gaze_csv):
            continue

        print(f"Processing {participant} / {session}")

        try:
            fixations = process_gaze_to_fixations(gaze_csv)

            if fixations is None:
                print("No fixations found")
                continue

            # --- SAVE PATH ---
            save_dir = os.path.join(output_root, participant)
            os.makedirs(save_dir, exist_ok=True)

            save_path = os.path.join(save_dir, f"{session}_fixations.csv")

            fixations.to_csv(save_path, index=False)

            print(f"Saved → {save_path}")

        except Exception as e:
            print(f"Error in {participant}/{session}: {e}")

Processing sub-OE020 / ses-Norrebro
Saved → /mnt/raid/emotional_data_raquel/fulldata_analysis_mine/fixations/sub-OE020/ses-Norrebro_fixations.csv
Processing sub-OE020 / ses-Nordhavn
Saved → /mnt/raid/emotional_data_raquel/fulldata_analysis_mine/fixations/sub-OE020/ses-Nordhavn_fixations.csv
Processing sub-OE005 / ses-Nordhavn
Saved → /mnt/raid/emotional_data_raquel/fulldata_analysis_mine/fixations/sub-OE005/ses-Nordhavn_fixations.csv
Processing sub-OE005 / ses-Hellerup
Saved → /mnt/raid/emotional_data_raquel/fulldata_analysis_mine/fixations/sub-OE005/ses-Hellerup_fixations.csv
Processing sub-OE018 / ses-Hellerup
Saved → /mnt/raid/emotional_data_raquel/fulldata_analysis_mine/fixations/sub-OE018/ses-Hellerup_fixations.csv
Processing sub-OE012 / ses-Norreport
Saved → /mnt/raid/emotional_data_raquel/fulldata_analysis_mine/fixations/sub-OE012/ses-Norreport_fixations.csv
Processing sub-OE021 / ses-Norrebro
Saved → /mnt/raid/emotional_data_raquel/fulldata_analysis_mine/fixations/sub-OE021/ses

Aligning fixations with video frames for all participants

In [4]:
def align_fixations(session_path, fixations):

    datapicker = create_datapicker(path=session_path, schema=build_schema)
    dataset = load_dataset(datapicker.selected_path, schema=build_schema)

    gaze = dataset.streams.PupilLabs.PupilGaze.data

    gaze = (
        gaze.groupby(level=0)[["GazeX", "GazeY"]]
        .mean()
        .rename(columns={"GazeX": "x", "GazeY": "y"})
    )

    decoded_frames = dataset.streams.PupilLabs.DecodedFrames.data
    decoded_frames = decoded_frames[decoded_frames.Value > 0]

    gaze = gaze.reindex(decoded_frames.index, method="nearest")

    gaze["frame"] = np.arange(len(gaze))

    aligned = pd.merge_asof(
        fixations.sort_values("mid_time"),
        gaze,
        left_on="mid_time",
        right_index=True,
        direction="nearest"
    )

    return aligned

In [8]:
# ===== LOAD CLIP =====
device = "cuda" if torch.cuda.is_available() else "cpu"

model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32",use_safetensors=True).to(device)
processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")

In [9]:
# ===== LOAD COCO LABELS =====
ann_file = "/home/s243636/master-thesis/notebooks/instances_val2017.json"

coco = COCO(ann_file)
cats = coco.loadCats(coco.getCatIds())

class_names = [c["name"] for c in cats]
labels = [f"a photo of a {name}" for name in class_names]

loading annotations into memory...
Done (t=0.54s)
creating index...
index created!


In [10]:
# ===== CLIP FUNCTION =====
def classify_crop(crop):
    image = Image.fromarray(cv.cvtColor(crop, cv.COLOR_BGR2RGB))

    inputs = processor(
        text=labels,
        images=image,
        return_tensors="pt",
        padding=True
    ).to(device)

    outputs = model(**inputs)
    probs = outputs.logits_per_image.softmax(dim=1)

    return class_names[probs.argmax().item()]

In [11]:
def process_video_with_clip(video_path, aligned):

    cap = cv.VideoCapture(video_path)
    results = []

    grouped = aligned.groupby("frame")

    for frame_id, gaze_rows in grouped:

        cap.set(cv.CAP_PROP_POS_FRAMES, int(frame_id))
        ret, frame = cap.read()
        if not ret:
            continue

        for _, gaze in gaze_rows.iterrows():
            x, y = int(gaze["x"]), int(gaze["y"])

            crop = frame[
                max(0, y-100):y+100,
                max(0, x-100):x+100
            ]

            label = classify_crop(crop)

            results.append({
                "frame": frame_id,
                "x": x,
                "y": y,
                "label": label
            })

    cap.release()
    return results

In [12]:
data_root = "/mnt/raid/emotional_data_raquel/data"
fix_root = "/mnt/raid/emotional_data_raquel/fulldata_analysis_mine/fixations"
output_root = "/mnt/raid/emotional_data_raquel/fulldata_analysis_mine/clip"

os.makedirs(output_root, exist_ok=True)

for participant in os.listdir(data_root):

    participant_path = os.path.join(data_root, participant)

    for session in os.listdir(participant_path):

        session_path = os.path.join(participant_path, session)
        participant = f"sub-{participant}"

        # --- LOAD FIXATIONS ---
        session_name = session.split("_")[1]
        fix_path = os.path.join(fix_root, participant, f"ses-{session_name}_fixations.csv")

        if not os.path.exists(fix_path):
            continue

        fixations = pd.read_csv(fix_path, parse_dates=["start_time", "end_time", "mid_time"])

        # --- ALIGN ---
        try:
            aligned = align_fixations(session_path, fixations)
        except Exception as e:
            print(f"Alignment error {participant}/{session}: {e}")
            continue

        # --- VIDEO ---
        video_path = os.path.join(session_path, "pupil_video.avi")

        if not os.path.exists(video_path):
            continue

        print(f"Processing {participant} / {session}")

        try:
            results = process_video_with_clip(video_path, aligned)
        except Exception as e:
            print(f"CLIP error {participant}/{session}: {e}")
            continue

        # --- SAVE ---
        save_dir = os.path.join(output_root, participant)
        os.makedirs(save_dir, exist_ok=True)

        save_path = os.path.join(save_dir, f"{session}_clip.csv")

        pd.DataFrame(results).to_csv(save_path, index=False)

        print(f"Saved → {save_path}")

Attempting to automatically correct eeg timestamps to harp timestamps...
Done.
Processing sub-OE011 / Copenhagen_Nordhavn_sub-OE203011_2024-06-25T113254Z
Saved → /mnt/raid/emotional_data_raquel/fulldata_analysis_mine/clip/sub-OE011/Copenhagen_Nordhavn_sub-OE203011_2024-06-25T113254Z_clip.csv
Processing sub-OE015 / Copenhagen_Norreport_sub-OE2002015_2024-06-28T081232Z
Saved → /mnt/raid/emotional_data_raquel/fulldata_analysis_mine/clip/sub-OE015/Copenhagen_Norreport_sub-OE2002015_2024-06-28T081232Z_clip.csv
Attempting to automatically correct eeg timestamps to harp timestamps...
Done.
Processing sub-OE019 / Copenhagen_Hellerup_sub-OE204019_2024-06-27T152050Z
Saved → /mnt/raid/emotional_data_raquel/fulldata_analysis_mine/clip/sub-OE019/Copenhagen_Hellerup_sub-OE204019_2024-06-27T152050Z_clip.csv
Attempting to automatically correct eeg timestamps to harp timestamps...
Done.
Processing sub-OE010 / Copenhagen_Nordhavn_sub-OE203010_2024-06-28T140152Z


KeyboardInterrupt: 